## 📌 Week 3: Sentiment Analysis & Priority Scoring

In this week, we focus on understanding the emotional tone of citizen complaints.

Objectives:
- Classify sentiment using AI (BERT model)
- Categorize complaints into Positive, Negative, Neutral
- Assign a priority score based on urgency

This helps government officials prioritize critical issues efficiently.

## 🔄 Step 1: Load Cleaned Dataset

Reload the cleaned dataset from Week 1.

In [2]:
import pandas as pd
df_clean = pd.read_csv('../data/processed/cleaned_data.csv')
# Basic cleaning
df_clean = df_clean.dropna(subset=['complaint', 'department'])
df_clean = df_clean[df_clean['complaint'].str.strip() != ""]
df_clean['complaint'] = df_clean['complaint'].astype(str)
df_clean.head()

,complaint,department
0,railways railway board miscellaneous railw...,MORLY
1,xaxpxxxxxxx\tregarding cbcid inspection closed...,GOVUP
2,labour and employment pf withdrawal others...,MOLBR
3,labour and employment pension others name...,MOLBR
4,xaxpxxxxxxx\tregarding cbcid inspection closed...,GOVUP


## ⚡ Step 2: Reduce Dataset Size
BERT models are computationally expensive, so we use a smaller sample for faster processing.

In [3]:
df_clean = df_clean.sample(500, random_state=42)

## 🤖 Step 3: Load Pre-trained BERT Model

We use a pre-trained transformer model for sentiment analysis.

In [5]:
from transformers import pipeline
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

## 🧪 Step 4: Test BERT Model

Before applying to dataset, test with sample text.

In [6]:
print(sentiment_pipeline("This is an urgent issue and needs immediate attention"))

[{'label': 'NEGATIVE', 'score': 0.9836799502372742}]


## 🔍 Step 5: Apply BERT to Complaints

We apply the model to each complaint to detect sentiment.

In [7]:
def bert_sentiment(text):
    try:
        result = sentiment_pipeline(text[:512])[0]
        return result['label']
    except:
        return "NEUTRAL"

df_clean['bert_sentiment'] = df_clean['complaint'].apply(bert_sentiment)
df_clean[['complaint', 'bert_sentiment']].head()

,complaint,bert_sentiment
134728,financial services insurance division nonset...,NEGATIVE
12575,land seeding no,NEGATIVE
131622,frank,POSITIVE
57373,financial services insurance division fraud ...,NEGATIVE
159244,kindly provide any kind of scholarship for ski...,POSITIVE


## 🔄 Step 6: Convert Model Output

BERT returns POSITIVE/NEGATIVE. We convert it to custom labels.

In [8]:
def map_sentiment(label):
    if label == "NEGATIVE":
        return "Negative"
    elif label == "POSITIVE":
        return "Positive"
    else:
        return "Neutral"

df_clean['final_sentiment'] = df_clean['bert_sentiment'].apply(map_sentiment)
df_clean[['bert_sentiment', 'final_sentiment']].head()

,bert_sentiment,final_sentiment
134728,NEGATIVE,Negative
12575,NEGATIVE,Negative
131622,POSITIVE,Positive
57373,NEGATIVE,Negative
159244,POSITIVE,Positive


## 🚨 Step 7: Assign Priority Score
Each complaint is assigned a priority score based on sentiment.
- Negative → High Priority
- Neutral → Medium Priority
- Positive → Low Priority

In [9]:
def get_priority(sentiment):
    if sentiment == "Negative":
        return 3
    elif sentiment == "Neutral":
        return 2
    else:
        return 1
df_clean['priority'] = df_clean['final_sentiment'].apply(get_priority)
df_clean[['complaint', 'final_sentiment', 'priority']].head()

,complaint,final_sentiment,priority
134728,financial services insurance division nonset...,Negative,3
12575,land seeding no,Negative,3
131622,frank,Positive,1
57373,financial services insurance division fraud ...,Negative,3
159244,kindly provide any kind of scholarship for ski...,Positive,1


## 📊 Step 8: Final Output
Now each complaint has:
- Sentiment
- Priority Score
This completes the sentiment analysis pipeline.

In [10]:
df_clean.head()

,complaint,department,bert_sentiment,final_sentiment,priority
134728,financial services insurance division nonset...,DEAID,NEGATIVE,Negative,3
12575,land seeding no,DOAAC,NEGATIVE,Negative,3
131622,frank,PMOPG,POSITIVE,Positive,1
57373,financial services insurance division fraud ...,DEAID,NEGATIVE,Negative,3
159244,kindly provide any kind of scholarship for ski...,DOSKD,POSITIVE,Positive,1
